# Appendix 8. Great Expectations 기초

## 1. Goal

이 notebook은 pandas DataFrame에 데이터 품질 규칙을 적용하는 Great Expectations(GX) Core의 기본 흐름을 익히는 자료입니다. Expectation 하나를 시험하는 단계에서 시작해 Expectation Suite, Validation Definition, Checkpoint로 검증 범위를 확장합니다.

합성 데이터의 의도적인 오류를 검출하며, 실패 결과를 숨기지 않고 어떤 규칙과 행이 실패했는지 읽는 데 집중합니다.

## 2. Setup

프로젝트에 설치된 GX 1.x와 pandas를 사용합니다. `mode="ephemeral"` Data Context를 사용하므로 GX project나 Data Docs를 repository에 만들지 않습니다. 실행 중인 DataFrame도 session 밖에 저장하지 않습니다.

In [ ]:
import great_expectations as gx
import pandas as pd
from great_expectations import expectations as gxe
from great_expectations.checkpoint import Checkpoint
from great_expectations.core.expectation_suite import ExpectationSuite
from great_expectations.core.validation_definition import ValidationDefinition

print({"great_expectations": gx.__version__, "pandas": pd.__version__})


## 3. Steps

### 3-1. Data Context와 Batch

#### 3-1-1. 검증할 DataFrame 준비

GX는 분석 결과를 대신 해석하는 도구가 아니라, 기대하는 데이터 상태를 실행 가능한 assertion으로 표현하는 도구입니다. 아래 DataFrame에는 duplicate ID, 결측값, 허용 범위를 벗어난 심박수와 target을 일부러 넣었습니다.

In [ ]:
measurements = pd.DataFrame(
    {
        "record_id": ["P101", "P102", "P102", "P104"],
        "heart_rate": [84.0, None, 92.0, 155.0],
        "target": [0, 1, 1, 2],
    }
)
measurements


#### 3-1-2. Data Source, Data Asset과 Batch Definition

Data Context는 GX 구성 객체를 관리합니다. Data Source는 pandas 같은 데이터 환경에 연결하고, Data Asset은 검증할 논리적 데이터 집합을 나타냅니다. Batch Definition은 그 Asset에서 어떤 레코드를 한 번의 검증 대상으로 가져올지 정합니다. in-memory DataFrame은 실행할 때 `batch_parameters`로 전달합니다.

In [ ]:
context = gx.get_context(mode="ephemeral")
data_source = context.data_sources.add_pandas(name="icu-data-source")
data_asset = data_source.add_dataframe_asset(name="measurements")
batch_definition = data_asset.add_batch_definition_whole_dataframe(
    name="whole-dataframe"
)
batch_parameters = {"dataframe": measurements}
batch = batch_definition.get_batch(batch_parameters=batch_parameters)

print(
    {
        "context_type": type(context).__name__,
        "asset_name": data_asset.name,
        "batch_definition_name": batch_definition.name,
    }
)


### 3-2. Expectation과 Expectation Suite

#### 3-2-1. Expectation 하나를 Batch에서 시험

Expectation은 데이터에 대해 검증 가능한 assertion 하나를 표현합니다. 먼저 심박수 결측값 규칙 하나만 실행해 결과 구조를 확인합니다. `success`는 전체 성공 여부이고, `result`에는 검사한 원소 수와 unexpected 값 요약이 들어 있습니다.

In [ ]:
not_null_expectation = gxe.ExpectColumnValuesToNotBeNull(
    column="heart_rate"
)
single_result = batch.validate(not_null_expectation).to_json_dict()
single_summary = {
    "success": single_result["success"],
    "element_count": single_result["result"]["element_count"],
    "unexpected_count": single_result["result"]["unexpected_count"],
    "partial_unexpected_list": single_result["result"][
        "partial_unexpected_list"
    ],
}
single_summary


#### 3-2-2. 관련 규칙을 Expectation Suite로 묶기

Expectation Suite는 같은 데이터 계약을 설명하는 여러 Expectation을 묶습니다. column 순서, ID uniqueness, 결측값, 값 범위와 허용 집합처럼 규칙의 의도가 드러나는 API를 선택합니다. `mostly`를 사용하면 일부 실패를 허용할 수 있지만, 허용 비율은 분석 결과가 아니라 versioned quality policy에서 정해야 합니다.

In [ ]:
expectations = [
    gxe.ExpectTableColumnsToMatchOrderedList(
        column_list=["record_id", "heart_rate", "target"]
    ),
    gxe.ExpectColumnValuesToBeUnique(column="record_id"),
    gxe.ExpectColumnValuesToNotBeNull(column="heart_rate"),
    gxe.ExpectColumnValuesToBeBetween(
        column="heart_rate", min_value=30, max_value=140
    ),
    gxe.ExpectColumnValuesToBeInSet(column="target", value_set=[0, 1]),
]

suite = ExpectationSuite(name="icu-measurements-suite")
for expectation in expectations:
    suite.add_expectation(expectation)
suite = context.suites.add(suite)

print({"suite_name": suite.name, "expectation_count": len(expectations)})


### 3-3. Validation과 Checkpoint

#### 3-3-1. Validation Definition 실행

Validation Definition은 Batch Definition과 Expectation Suite를 연결합니다. 실행 결과의 `statistics`로 전체 규칙 수와 성공·실패 수를 빠르게 확인한 뒤, 실패한 Expectation별 `result`를 내려가며 원인을 찾습니다.

In [ ]:
validation_definition = context.validation_definitions.add(
    ValidationDefinition(
        name="icu-measurements-validation",
        data=batch_definition,
        suite=suite,
    )
)
validation_result = validation_definition.run(
    batch_parameters=batch_parameters
)
validation_json = validation_result.to_json_dict()

print(validation_json["statistics"])
failed_expectations = [
    {
        "expectation_type": item["expectation_config"]["type"],
        "unexpected_count": item["result"].get("unexpected_count"),
    }
    for item in validation_json["results"]
    if not item["success"]
]
failed_expectations


#### 3-3-2. Checkpoint로 반복 실행 경계 만들기

Checkpoint는 하나 이상의 Validation Definition을 실행하고 결과에 따라 Action을 연결할 수 있는 운영 경계입니다. 여기서는 외부 Action 없이 local validation만 실행합니다. 실제 pipeline에서는 결과 저장, Data Docs 생성, 알림 같은 side effect를 adapter에서 명시적으로 구성합니다.

In [ ]:
checkpoint = context.checkpoints.add(
    Checkpoint(
        name="icu-measurements-checkpoint",
        validation_definitions=[validation_definition],
    )
)
checkpoint_result = checkpoint.run(batch_parameters=batch_parameters)
print(
    {
        "checkpoint_success": checkpoint_result.success,
        "validation_count": len(checkpoint_result.run_results),
    }
)


## 4. Checks

의도적으로 잘못 만든 데이터가 예상한 규칙에서 실패했는지 확인합니다. 검증 실패는 notebook 실패가 아니라 데이터 계약 위반을 정상적으로 찾아낸 결과입니다.

In [ ]:
assert single_summary["unexpected_count"] == 1
assert validation_json["statistics"]["evaluated_expectations"] == 5
assert validation_json["statistics"]["unsuccessful_expectations"] == 4
assert len(failed_expectations) == 4
assert checkpoint_result.success is False
assert len(checkpoint_result.run_results) == 1
print("All appendix checks passed.")


## 5. Next Steps

- quality rule의 threshold와 허용 집합은 Python에 중복 작성하지 않고 versioned contract에서 가져옵니다.
- 실패한 전체 row가 필요하면 result format과 unexpected row 조회 범위를 명시합니다.
- notebook에서 검증한 Suite를 pipeline adapter의 Validation Definition과 Checkpoint로 옮깁니다.
- file-backed Context와 Data Docs는 생성 위치를 artifact 경계로 제한하고 Git에 generated output을 넣지 않습니다.

## 6. References

이 notebook의 설명과 예제는 다음 공식 문서를 기준으로 작성했습니다.

- [Connect to dataframe data](https://docs.greatexpectations.io/docs/core/connect_to_data/dataframes/)
- [Define Expectations](https://docs.greatexpectations.io/docs/core/define_expectations/)
- [Create an Expectation](https://docs.greatexpectations.io/docs/core/define_expectations/create_an_expectation/)
- [Run Validations](https://docs.greatexpectations.io/docs/core/run_validations/)
- [Run a Checkpoint](https://docs.greatexpectations.io/docs/core/trigger_actions_based_on_results/run_a_checkpoint/)